In [204]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [205]:
pokazaniya_df = pd.read_csv('pokazaniya.csv', sep=';', parse_dates=['data'])
sobytiya_df = pd.read_csv('sobytiya.csv', sep=';', parse_dates=['data'])
pribory_df = pd.read_csv('pribory.csv', sep=';', parse_dates=['mesyac_vypuska'], date_format='%Y-%m')


In [206]:
pokazaniya_df = pokazaniya_df.sort_values(['anon_id', 'data'], ascending=[True, True], ignore_index=True)
sobytiya_df = sobytiya_df.sort_values(['anon_id', 'data'], ascending=[True, True], ignore_index=True)
pribory_df = pribory_df.sort_values(['anon_id', 'mesyac_vypuska'], ascending=[True, True], ignore_index=True)

### Задание 1
***Отрицательный расход (счётчик «пошёл назад»): критерий, сколько приборов, примеры ID, что это значит.***


In [207]:
pokazaniya_df.pokazanie.describe()

count    6.934430e+05
mean     1.527827e+02
std      8.729499e+03
min      0.000000e+00
25%      1.960000e+00
50%      4.758583e+01
75%      1.393205e+02
max      4.273565e+06
Name: pokazanie, dtype: float64

In [208]:
pokazaniya_df['pred_pokazanie'] = pokazaniya_df.groupby('anon_id')['pokazanie'].shift(1)
pokazaniya_df['rashod'] = pokazaniya_df['pokazanie'] - pokazaniya_df['pred_pokazanie']
result = pokazaniya_df.query('rashod < 0')['anon_id'].nunique()
print(f'Критерий: отрицательный расход rashod < 0 означает, что показание счётчика уменьшилось. Таких приборов {result}.')

Критерий: отрицательный расход rashod < 0 означает, что показание счётчика уменьшилось. Таких приборов 314.


Почему именно -1e-3? потому что проанализировал по счетчикам и как правило на счетчиках после точки идут до 3 цифр.
А все что идет после 3 цифр это мелкие изменения, которые можно считать погрешностями

In [209]:
negative_df = pokazaniya_df[
    pokazaniya_df['rashod'] < -1e-3
]
negative_devices = negative_df['anon_id'].nunique()
negative_devices
example_ids = (
    negative_df
    .nsmallest(5, 'rashod')['anon_id']
    .unique()
    .tolist()
)
print(f'Количество таких приборов: {negative_devices}')
print(f'Примеры ID: {example_ids}')

Количество таких приборов: 102
Примеры ID: [2519659, 2491162, 8908023, 2159987]


### Задание 2
***Всплески расхода за сутки: абсолютный и относительный порог, сколько, примеры ID.***


2.1 Рассчитываем количество дней между показаниями и суточный расход

In [210]:
pokazaniya_df['pred_data'] = pokazaniya_df.groupby('anon_id')['data'].shift(1)
pokazaniya_df['dney'] = (pokazaniya_df['data'] - pokazaniya_df['pred_data']).dt.days
pokazaniya_df['sutochniy_rashod'] = np.where(
    pokazaniya_df['dney'] > 0,
    pokazaniya_df['rashod'] / pokazaniya_df['dney'],
    np.nan
)
filtred_pokazaniya_df = pokazaniya_df[
    (pokazaniya_df['dney'] == 1) &
    (pokazaniya_df['rashod'] > 0)
].copy()

2.2 Абсолютный порог

In [211]:
Q1 = filtred_pokazaniya_df['sutochniy_rashod'].quantile(0.25)
Q3 = filtred_pokazaniya_df['sutochniy_rashod'].quantile(0.75)
IQR = Q3 - Q1
right = Q3 + 3 * IQR

abs_users = filtred_pokazaniya_df[
    filtred_pokazaniya_df['sutochniy_rashod'] > right
].copy()

top_abs_ids = (abs_users.nlargest(5, 'sutochniy_rashod')['anon_id'].unique().tolist())

filtred_pokazaniya_df['srednee_sutki'] = (filtred_pokazaniya_df.groupby('anon_id')['sutochniy_rashod'].transform('mean'))

rel_spikes = filtred_pokazaniya_df[filtred_pokazaniya_df['sutochniy_rashod'] > 3 * filtred_pokazaniya_df['srednee_sutki']].copy()

rel_spikes['ratio'] = (rel_spikes['sutochniy_rashod'] / rel_spikes['srednee_sutki'])

top_rel_ids = (rel_spikes.nlargest(5, 'ratio')['anon_id'].unique().tolist())

print(f'Абсолютный порог: выше {right.round(3)} м³/сутки')
print(f'Количество приборов: {abs_users["anon_id"].nunique()}')
print(f'Примеры ID: {top_abs_ids} \n')

print('Относительный порог: выше личного среднего в 3 раза')
print(f'Количество приборов: {rel_spikes["anon_id"].nunique()}')
print(f'Примеры ID: {top_rel_ids}')

Абсолютный порог: выше 0.788 м³/сутки
Количество приборов: 522
Примеры ID: [2491162, 8908023, 2159987, 7891133, 8973572] 

Относительный порог: выше личного среднего в 3 раза
Количество приборов: 1459
Примеры ID: [8908023, 6316651, 2569627, 2983464, 8148916]


### Задание 3
***Нереальные значения (физически невозможный расход): как нашли, какие приборы.***


In [212]:
merged_df = pokazaniya_df.merge(pribory_df, on='anon_id', how='left')
merged_df['limit'] = merged_df['model'].apply(
    lambda x: np.nan if pd.isna(x) else (120 if 'D20' in x else 70)
)
unrealistic_df = merged_df[merged_df['limit'].notna() & (merged_df['sutochniy_rashod'] > merged_df['limit'])].copy()
unrealistic_devices_count = unrealistic_df['anon_id'].nunique()
top_unrealistic_ids = (unrealistic_df.nlargest(5, 'sutochniy_rashod')['anon_id'].unique().tolist())
models_breakdown = (unrealistic_df['model'].value_counts(dropna=False).to_dict())

print(f'Найдено приборов с нереальным расходом: {unrealistic_devices_count}')
print(f'Примеры ID: {top_unrealistic_ids}')
print(f'Распределение по моделям: {models_breakdown}')

Найдено приборов с нереальным расходом: 54
Примеры ID: [2491162, 8908023, 2519659, 2159987]
Распределение по моделям: {'АКВА L110 D15 B E': 215, 'АКВА L110 D15 C E': 70, 'АКВА L130 D20 B E': 1}


### Задание 4
***Утечка (непрерывное потребление без «нулевых» суток): ваш критерий и количество.***


In [213]:
leak_df = pokazaniya_df[(pokazaniya_df['dney'] == 1) &(pokazaniya_df['sutochniy_rashod'] > 0)].copy()
leak_df['new_group'] = ~(leak_df.groupby('anon_id')['data'].diff().dt.days.eq(1))
leak_df['leak_group'] = (leak_df.groupby('anon_id')['new_group'].cumsum())
leak_counts = (leak_df.groupby(['anon_id', 'leak_group']).size().reset_index(name='leak_days'))
leaks = leak_counts[leak_counts['leak_days'] >= 30]

print(f'Количество приборов с утечкой: 'f'{leaks["anon_id"].nunique()}')
print(f'Примеры ID: 'f'{leaks.nlargest(5, "leak_days")["anon_id"].tolist()}')

Количество приборов с утечкой: 1168
Примеры ID: [1968874, 2931056, 7496751, 6554606, 1282534]


### Задание 5
***Нулевое потребление при идущих пакетах: порог и количество.***


In [214]:
zero_df = pokazaniya_df[(pokazaniya_df['dney'] == 1) &(pokazaniya_df['rashod'] == 0) &(pokazaniya_df['paketov_za_sutki'] > 0)].copy()
zero_df['new_group'] = ~(zero_df.groupby('anon_id')['data'].diff().dt.days.eq(1))
zero_df['zero_group'] = (zero_df.groupby('anon_id')['new_group'].cumsum()
)
zero_counts = (zero_df.groupby(['anon_id', 'zero_group']).size().reset_index(name='zero_days'))
zeros = zero_counts[zero_counts['zero_days'] >= 30]

print(f'Количество приборов с нулевым потреблением: 'f'{zeros["anon_id"].nunique()}')
print(f'Примеры ID: 'f'{zeros.nlargest(5, "zero_days")["anon_id"].tolist()}')

Количество приборов с нулевым потреблением: 694
Примеры ID: [7558835, 9686922, 9354769, 5374718, 1547698]


### Задание 6
***Ступенька (резкое устойчивое изменение уровня потребления): критерий и количество.***


In [215]:
valid_df = pokazaniya_df[(pokazaniya_df['dney'] == 1) & (pokazaniya_df['sutochniy_rashod'] >= 0)].copy()
periods = (valid_df.groupby('anon_id').agg(first_date=('data', 'min'), last_date=('data', 'max')))
periods['mid_date'] = (periods['first_date'] +(periods['last_date'] - periods['first_date']) / 2)
valid_df = valid_df.join(periods['mid_date'],on='anon_id')
valid_df['half'] = (valid_df['data'] > valid_df['mid_date']).astype(int)
step_stats = (valid_df.groupby(['anon_id', 'half'])['sutochniy_rashod'].median().unstack())
step_stats = step_stats.dropna()
step_stats = step_stats[(step_stats[0] > 0) &(step_stats[1] > 0)].copy()
step_stats['ratio'] = (step_stats[1] / step_stats[0]
)

step_stats['diff'] = (step_stats[1] - step_stats[0]
).abs()

step_anomalies = step_stats[((step_stats['ratio'] >= 2.0) | (step_stats['ratio'] <= 0.5)) &(step_stats['diff'] >= 0.1)]
print(f'Количество приборов со ступенькой: 'f'{len(step_anomalies)}')
print(f'Рост: 'f'{((step_stats["ratio"] >= 2.0) & (step_stats["diff"] >= 0.1)).sum()}')
print(f'Падение: 'f'{((step_stats["ratio"] <= 0.5) & (step_stats["diff"] >= 0.1)).sum()}')
print(f'Примеры ID: 'f'{step_anomalies.nlargest(5, "diff").index.tolist()}')

Количество приборов со ступенькой: 89
Рост: 50
Падение: 39
Примеры ID: [1581018, 4918146, 3743114, 7629899, 1171145]


### Задание 7
***Связь аномалий/падения расхода с событиями «магнит»: что обнаружили.***


In [216]:
mag_events = sobytiya_df[sobytiya_df['sobytie'].str.contains('Магнит|магнит', case=False, na=False)
][['anon_id', 'data']].drop_duplicates(['anon_id', 'data'])

mag_merged = pokazaniya_df[
    pokazaniya_df['dney'] == 1
].merge(
    mag_events,
    on='anon_id',
    suffixes=('', '_mag')
)

mag_merged['days_to_mag'] = (
    mag_merged['data'] - mag_merged['data_mag']
).dt.days

before = (
    mag_merged[
        (mag_merged['days_to_mag'] >= -14) &
        (mag_merged['days_to_mag'] < 0)
    ]
    .groupby(['anon_id', 'data_mag'])['sutochniy_rashod']
    .mean()
)

after = (
    mag_merged[
        (mag_merged['days_to_mag'] > 0) &
        (mag_merged['days_to_mag'] <= 14)
    ]
    .groupby(['anon_id', 'data_mag'])['sutochniy_rashod']
    .mean()
)

comp = pd.DataFrame({
    'before': before,
    'after': after
}).dropna()

comp['drop_ratio'] = (comp['after'] / comp['before'])

print(f'Всего приборов с магнитом (где есть история до/после): 'f'{comp.index.get_level_values("anon_id").nunique()}')
print(f'Из них расход упал в ноль: 'f'{comp[comp["after"] == 0].index.get_level_values("anon_id").nunique()}')
print(f'Из них расход упал минимум вдвое: 'f'{comp[comp["drop_ratio"] <= 0.5].index.get_level_values("anon_id").nunique()}')

Всего приборов с магнитом (где есть история до/после): 549
Из них расход упал в ноль: 186
Из них расход упал минимум вдвое: 193


### Задание 8
***2–3 собственные находки по потреблению.***


In [217]:
device_medians = (
    filtred_pokazaniya_df
    .groupby('anon_id')['sutochniy_rashod']
    .median()
)

median_threshold = device_medians.quantile(0.95)
high_median = device_medians[device_medians > median_threshold]

print(f'Порог: 95-й квантиль медианного суточного расхода = 'f'{median_threshold.round(3)} м³/сутки')

print(f'Количество приборов: {high_median.size}')
print(f'Примеры ID: {high_median.nlargest(5).index.tolist()}')

Порог: 95-й квантиль медианного суточного расхода = 0.415 м³/сутки
Количество приборов: 133
Примеры ID: [9277360, 4945470, 5927111, 1648954, 2803951]


In [218]:
cv_df = (filtred_pokazaniya_df.groupby('anon_id')['sutochniy_rashod'].agg(['mean', 'std']))

cv_df['cv'] = (cv_df['std'] /cv_df['mean'].replace(0, np.nan))

cv_threshold = cv_df['cv'].quantile(0.95)

high_cv = cv_df[cv_df['cv'] > cv_threshold]

print(f'Порог: 95-й квантиль коэффициента вариации = 'f'{cv_threshold.round(3)}')
print(f'Количество приборов: {high_cv.index.size}')
print(f'Примеры ID: {high_cv.nlargest(5, "cv").index.tolist()}')

Порог: 95-й квантиль коэффициента вариации = 1.781
Количество приборов: 122
Примеры ID: [8908023, 6316651, 2569627, 8148916, 2983464]
